# LSTM input preparation, Optuna optimization, and training

This notebook replaces the older LSTM input, Optuna, and final-training scripts. It prepares compact point inventories for ASCAT and SMAP, runs the real hyperparameter search, and trains the ten-seed ensemble.

The LSTM and FNO use the same seven static predictors, past-only windows of 32 valid satellite observations with scaled time gaps, fixed temporal validation/holdout dates, 80/20 spatial point split, pixel-balanced loss, static-feature masking/noise, gradient clipping, and a validation objective weighted 70% toward spatial transfer. Only their architectures and architecture-specific search spaces differ. No figures are generated. Each product is saved as one prediction-ready ensemble bundle containing only the hyperparameters, scaler, and trained weights needed for prediction.

In [ ]:
import os

from config import configure_runtime

configure_runtime()

import pandas as pd
import torch
from IPython.display import display

## 1. Paths and reusable imports

In [ ]:
from config import (
    LSTM_TRAINING_WORKERS,
    base_FP,
    cpuserver_data,
    das_FP,
    george_FP,
    nas_FP,
)
from LSTM.inputs import generate_input_csv
from LSTM.settings import (
    AREAS,
    CONTEXT_END_DATE,
    CONTEXT_START_DATE,
    MODEL_HOLDOUT_START_DATE,
    MODEL_SEEDS,
    PRODUCTS,
    STATIC_FEATURES,
    VALIDATION_START_DATE,
    WINDOW_SIZE,
)
from LSTM.training import (
    TRAINING_RECIPE_VERSION,
    optimize_hyperparameters,
    save_ensemble_bundle,
    train_ensemble,
)

from config import figures_FP, results_FP


## 2. Scientific and runtime settings

The study calendar is 2015-04-01 through 2023-12-31. Training targets stop before 2021-06-15, temporal validation spans 2021-06-15 through 2022-04-20, and later observations remain held out. Static features are fitted only on the deterministic 80% spatial-training subset. Training and validation give each represented pixel equal weight. Both Optuna and final training use 5% static-feature masking, Gaussian noise with standard deviation 0.05 on retained features, and gradient clipping at 1. Optuna uses seed 42; final training uses all ten listed seeds.

In [ ]:
DATA_PRODUCTS = PRODUCTS
INPUT_AREAS = AREAS
MIN_VALID_OBSERVATIONS = 100
N_OPTUNA_TRIALS = 200
OPTUNA_EPOCHS = 50
FINAL_TRAINING_EPOCHS = 500
FINAL_MODEL_SEEDS = MODEL_SEEDS
TRAINING_WORKERS = LSTM_TRAINING_WORKERS
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

# False keeps only necessary final artifacts. Set True to retain resumable studies.
SAVE_OPTUNA_DATABASES = False

print(f"Device: {DEVICE}")
print(f"Final-training workers: {TRAINING_WORKERS}")
print(f"Static features ({len(STATIC_FEATURES)}): {list(STATIC_FEATURES)}")
print(f"Observation window: {WINDOW_SIZE} valid SSM events")
print(
    f"Calendar/splits: {CONTEXT_START_DATE}..{CONTEXT_END_DATE}; "
    f"validation={VALIDATION_START_DATE}; holdout={MODEL_HOLDOUT_START_DATE}"
)

In [ ]:
ISMN_RESULT_FP = os.path.join(results_FP, "ISMN"
)
MODEL_INPUT_FILE = os.path.join(results_FP, "CONUS_Prediction", "Model_input_static_eqd_010.nc"
)
LSTM_RESULT_FP = os.path.join(results_FP, "LSTM"
)
TRAIN_RESULT_FP = os.path.join(LSTM_RESULT_FP, "Train")
os.makedirs(TRAIN_RESULT_FP, exist_ok=True)

INPUT_CSV = {
    (area, product): os.path.join(
        LSTM_RESULT_FP, area, f"LSTM_input_{product}.csv"
    )
    for area in INPUT_AREAS
    for product in DATA_PRODUCTS
}
BUNDLE_FILE = {
    product: os.path.join(TRAIN_RESULT_FP, f"LSTM_{product}_ensemble.pt")
    for product in DATA_PRODUCTS
}

if not os.path.exists(MODEL_INPUT_FILE):
    raise FileNotFoundError(f"Missing static model input: {MODEL_INPUT_FILE}")

## 3. Prepare the necessary point/static input CSV files

The CSVs are compact pixel inventories, not copies of the time series. Dynamic SSM and observed RZSM remain in the point NetCDF files created by `ISMN_preprocessing.ipynb`. Both products use the same seven static predictors; product-specific valid-observation counts are retained.

In [ ]:
input_summaries = []

for area in INPUT_AREAS:
    for product in DATA_PRODUCTS:
        input_frame = generate_input_csv(
            product=product,
            area=area,
            model_input_file=MODEL_INPUT_FILE,
            ismn_root=ISMN_RESULT_FP,
            output_file=INPUT_CSV[(area, product)],
            min_valid_observations=MIN_VALID_OBSERVATIONS,
        )
        input_summaries.append(
            {
                "area": area,
                "product": product,
                "pixels": 0 if input_frame is None else len(input_frame),
                "file": INPUT_CSV[(area, product)],
            }
        )

display(pd.DataFrame(input_summaries))

## 4. Optimize hyperparameters

This is the expensive Optuna stage. The default in-memory studies create no database files. Set `SAVE_OPTUNA_DATABASES = True` before running this cell only when interruption recovery is worth retaining two SQLite files.

In [ ]:
optimization_results = {}

for product in DATA_PRODUCTS:
    storage = None
    if SAVE_OPTUNA_DATABASES:
        database_file = os.path.join(
            TRAIN_RESULT_FP,
            f"LSTM_{product}_{TRAINING_RECIPE_VERSION}_optuna.db",
        )
        storage = f"sqlite:///{database_file}"

    best_parameters, scaler, study = optimize_hyperparameters(
        product=product,
        input_csv=INPUT_CSV[("Train", product)],
        ismn_root=ISMN_RESULT_FP,
        n_trials=N_OPTUNA_TRIALS,
        num_epochs=OPTUNA_EPOCHS,
        device=DEVICE,
        storage=storage,
    )
    optimization_results[product] = {
        "best_parameters": best_parameters,
        "scaler": scaler,
        "study": study,
    }
    print(product, best_parameters)

## 5. Train and save the final ten-seed ensembles

Each output bundle is the only trained-model artifact needed by `LSTM_prediction.ipynb`. It contains the selected hyperparameters, training-only scaler, and all ten state dictionaries. Training-loss figures and version/provenance records are intentionally omitted.

In [ ]:
training_summaries = []

for product in DATA_PRODUCTS:
    optimized = optimization_results[product]
    ensemble = train_ensemble(
        product=product,
        input_csv=INPUT_CSV[("Train", product)],
        ismn_root=ISMN_RESULT_FP,
        scaler=optimized["scaler"],
        hyperparameters=optimized["best_parameters"],
        seeds=FINAL_MODEL_SEEDS,
        device=DEVICE,
        num_epochs=FINAL_TRAINING_EPOCHS,
        workers=TRAINING_WORKERS,
    )
    save_ensemble_bundle(
        output_file=BUNDLE_FILE[product],
        ensemble=ensemble,
        scaler=optimized["scaler"],
        hyperparameters=optimized["best_parameters"],
    )
    training_summaries.append(
        {
            "product": product,
            "models": len(ensemble),
            "bundle": BUNDLE_FILE[product],
        }
    )

display(pd.DataFrame(training_summaries))